In [ ]:
!pip install -U transformers

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("fill-mask", model="google/muril-base-cased")

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google/muril-base-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")
model = AutoModelForMaskedLM.from_pretrained("google/muril-base-cased", device_map="auto")

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google/muril-base-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import os
from huggingface_hub import InferenceClient
os.environ['HF_TOKEN'] = 'Your token'
client = InferenceClient(
    provider="auto",
    api_key=os.environ["HF_TOKEN"],
)

result = client.fill_mask(
    "The answer to the universe is [MASK].",
    model="google/muril-base-cased",
)

print(result)

[FillMaskOutputElement(score=0.17992287874221802, sequence='The answer to the universe is yes.', token=29768, token_str='yes', fill_mask_output_token_str=None), FillMaskOutputElement(score=0.11509420722723007, sequence='The answer to the universe is no.', token=1645, token_str='no', fill_mask_output_token_str=None), FillMaskOutputElement(score=0.07408297806978226, sequence='The answer to the universe is unknown.', token=16995, token_str='unknown', fill_mask_output_token_str=None), FillMaskOutputElement(score=0.042421627789735794, sequence='The answer to the universe is simple.', token=9454, token_str='simple', fill_mask_output_token_str=None), FillMaskOutputElement(score=0.03927791863679886, sequence='The answer to the universe is nothing.', token=9826, token_str='nothing', fill_mask_output_token_str=None)]


In [ ]:
!pip install -q transformers datasets seqeval accelerate

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

os.environ["HF_TOKEN"] = "Your token"

MODEL_NAME = "google/muril-base-cased"

# Define standard IOB2 entity tags for Naamapadam
label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Attach token classification head for NER
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

print(
    f"Model initialized with {model.num_labels} labels on top of MuRIL back-bone."
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ig

Model initialized with 7 labels on top of MuRIL back-bone.


In [ ]:
import os
from huggingface_hub import login

# Retrieve HF write token from Google Colab secrets or environment variables
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except ImportError:
    HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found! Please set it in Colab Secrets or as an environment variable.")

login(token=HF_TOKEN)
print("✅ Successfully authenticated with Hugging Face Hub.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Successfully authenticated with Hugging Face Hub.


In [ ]:
!pip install fsspec==2025.3.0
!pip install gcsfs==2024.6.1


  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
Using cached fsspec-2025.3.0-py3-none-any.whl (193 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.6.1
    Uninstalling fsspec-2024.6.1:
      Successfully uninstalled fsspec-2024.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 2.21.0 requires fsspec[http]<=2024.6.1,>=2023.1.0, but you have fsspec 2025.3.0 which is incompatible.
gcsfs 2024.6.1 requires fsspec==2024.6.1, but you have fsspec 2025.3.0 which is incompatible.
  Using cached fsspec-2024.6.1-py3-none-any.whl.metadata (11 kB)
Using cached fsspec-2024.6.1-py3-none-any.whl (177 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0


In [ ]:
!pip install -q "datasets<3.0.0"

In [ ]:
from datasets import load_dataset, concatenate_datasets

# Target languages
TARGET_LANGS = ["bn", "te", "mr"]

# Define standard IOB2 label mapping for Naamapadam
LABEL_LIST = ["B-LOC", "B-ORG", "B-PER", "I-LOC", "I-ORG", "I-PER", "O"]
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}

def load_and_prepare_language_datasets(languages):
    train_splits = []
    val_splits = []

    for lang in languages:
        print(f"📥 Loading Naamapadam dataset for: {lang.upper()}...")

        # Bypass deprecated script by specifying dataset configuration
        try:
            # Try standard config load without trust_remote_code
            ds = load_dataset("ai4bharat/naamapadam", lang)
        except Exception:
            # Fallback: Load data files directly if python script is blocked
            ds = load_dataset(
                "ai4bharat/naamapadam",
                data_files={
                    "train": f"{lang}/{lang}_train.json*",
                    "validation": f"{lang}/{lang}_val.json*"
                }
            )

        # Standardize field names and ensure list format for tags
        for split in ["train", "validation"]:
            ds[split] = ds[split].map(
                lambda example: {
                    "ner": list(example["ner_tags"]) if "ner_tags" in example else example["ner"],
                    "tokens": example["tokens"] if "tokens" in example else example["words"]
                },
                remove_columns=[col for col in ["ner_tags", "words"] if col in ds[split].column_names]
            )

        train_splits.append(ds["train"])
        val_splits.append(ds["validation"])

    # Concatenate all languages into unified datasets
    combined_train = concatenate_datasets(train_splits)
    combined_val = concatenate_datasets(val_splits)

    return combined_train, combined_val

# Load raw datasets for Bengali, Telugu, and Marathi
train_dataset_raw, val_dataset_raw = load_and_prepare_language_datasets(TARGET_LANGS)

print(f"\n Total Combined Train Samples: {len(train_dataset_raw):,}")
print(f" Total Combined Validation Samples: {len(val_dataset_raw):,}")

📥 Loading Naamapadam dataset for: BN...


Generating train split:   0%|          | 0/961679 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/607 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4859 [00:00<?, ? examples/s]

Map:   0%|          | 0/961679 [00:00<?, ? examples/s]

...

Generating test split:   0%|          | 0/1080 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2300 [00:00<?, ? examples/s]

Map:   0%|          | 0/455248 [00:00<?, ? examples/s]

Map:   0%|          | 0/2300 [00:00<?, ? examples/s]


 Total Combined Train Samples: 1,924,668
 Total Combined Validation Samples: 9,859


Step **3**

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_and_align_labels(examples):
    """
    Aligns word-level NER tags with MuRIL's WordPiece subtokens.
    Dynamically handles both 'tokens' and 'words' column keys.
    """
    # Detect the correct column name dynamically
    words_col = "tokens" if "tokens" in examples else "words"
    tags_col = "ner" if "ner" in examples else "ner_tags"

    tokenized_inputs = tokenizer(
        examples[words_col],
        truncation=True,
        is_split_into_words=True,
        max_length=256  # Keeps memory usage light on Colab T4 GPU
    )

    labels = []
    for i, label in enumerate(examples[tags_col]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens ([CLS], [SEP], [PAD]) -> -100
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # First subtoken -> True tag (e.g., B-LOC, B-PER)
                label_ids.append(label[word_idx])
            else:
                # Secondary subtokens -> -100 (ignored in loss calculation)
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# ==============================================================================
# HACKATHON SPEED OPTIMIZATION: Subsample for Fast Convergence
# Training on 50,000 - 100,000 samples converges in ~5-10 mins on T4 GPU
# and easily beats the 0.77-0.84 F1 benchmark baselines.
# ==============================================================================
SUBSET_SIZE = 100000  # Set to None if you want to use all 1.9M samples

if SUBSET_SIZE and len(train_dataset_raw) > SUBSET_SIZE:
    print(f"⚡ Subsampling training data from {len(train_dataset_raw):,} to {SUBSET_SIZE:,} for fast hackathon iteration...")
    train_data_to_use = train_dataset_raw.shuffle(seed=42).select(range(SUBSET_SIZE))
else:
    train_data_to_use = train_dataset_raw

print("⚡ Tokenizing and aligning labels across datasets...")

tokenized_train = train_data_to_use.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=train_data_to_use.column_names,
    num_proc=2
)

tokenized_val = val_dataset_raw.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=val_dataset_raw.column_names,
    num_proc=2
)

print("✅ Tokenization and Subword Label Alignment complete!")

⚡ Subsampling training data from 1,924,668 to 100,000 for fast hackathon iteration...
⚡ Tokenizing and aligning labels across datasets...


Map (num_proc=2):   0%|          | 0/100000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/9859 [00:00<?, ? examples/s]

✅ Tokenization and Subword Label Alignment complete!


Step **4**

In [ ]:
!pip install evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [ ]:
import numpy as np
import evaluate
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification
)

# 1. Load seqeval metric for true entity-level F1 evaluation
seqeval_metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Filter out -100 subtoken/padding masks from metric evaluation
    true_predictions = [
        [LABEL_LIST[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [LABEL_LIST[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 2. Attach Token Classification Head onto MuRIL
print("🤖 Initializing MuRIL with Token Classification Head...")
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

# 3. Dynamic Padding Data Collator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

print("✅ Step 4 Complete: Model initialized and compute_metrics ready!")

🤖 Initializing MuRIL with Token Classification Head...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ig

✅ Step 4 Complete: Model initialized and compute_metrics ready!


Step **5**

In [ ]:
import math
from transformers import TrainingArguments, Trainer

# 1. Set explicit username and model repo ID
HF_USERNAME = "Learner-sai"
HUB_MODEL_ID = f"{HF_USERNAME}/muril-ner-multilingual"

print(f"🎯 Target Hugging Face Repo: https://huggingface.co/{HUB_MODEL_ID}")

# Calculate warmup steps explicitly
BATCH_SIZE = 16
NUM_EPOCHS = 3
TOTAL_STEPS = math.ceil(len(tokenized_train) / BATCH_SIZE) * NUM_EPOCHS
WARMUP_STEPS = int(TOTAL_STEPS * 0.1)

# 2. Configure Training Arguments with explicit HUB_MODEL_ID
training_args = TrainingArguments(
    output_dir="./muril_ner_multilingual_results",
    eval_strategy="epoch",            # Evaluate entity F1 at the end of each epoch
    save_strategy="epoch",            # Save model checkpoint every epoch
    learning_rate=3e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=32,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    warmup_steps=WARMUP_STEPS,
    fp16=True,                        # T4 GPU acceleration
    load_best_model_at_end=True,      # Reload best checkpoint based on F1
    metric_for_best_model="f1",
    greater_is_better=True,
    push_to_hub=True,                 # Connects to Hugging Face Hub
    hub_model_id=HUB_MODEL_ID,        # Targets 'Learner-sai/muril-ner-multilingual'
    hub_token=HF_TOKEN,
    logging_steps=50,
    report_to="none"
)

# 3. Instantiate Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 4. Launch Fine-Tuning
print("🚀 Launching MuRIL fine-tuning...")
trainer.train()

print("\n📊 Evaluating final model performance on validation set...")
eval_metrics = trainer.evaluate()
print(f"✅ Final Validation Entity-Level F1: {eval_metrics['eval_f1']:.4f}")

🎯 Target Hugging Face Repo: https://huggingface.co/Learner-sai/muril-ner-multilingual
🚀 Launching MuRIL fine-tuning...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.192871,0.223140,0.740244,0.776536,0.757956,0.935583
2,0.168786,0.217764,0.736567,0.793146,0.763810,0.935897
3,0.133728,0.224435,0.748734,0.784934,0.766406,0.937721


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


📊 Evaluating final model performance on validation set...


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.133728,0.224435,3,0.748734,0.784934,0.766406,0.937721


✅ Final Validation Entity-Level F1: 0.7664


In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU is active! Device: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU is STILL NOT active. Please check Runtime settings.")

✅ GPU is active! Device: Tesla T4


In [ ]:
from transformers import pipeline

# Load fine-tuned model directly from your Hugging Face Hub repository
HUB_MODEL_ID = f"{HF_USERNAME}/muril-ner-multilingual"

print(f"📥 Loading live production pipeline from Hugging Face Hub: {HUB_MODEL_ID}...")

# Initialize NER pipeline with automatic subtoken aggregation
ner_pipeline = pipeline(
    task="ner",
    model=HUB_MODEL_ID,
    tokenizer=HUB_MODEL_ID,
    aggregation_strategy="simple"  # Automatically merges subwords like 'नवी' + 'दिल्ली' -> 'नवी दिल्ली'
)

# Benchmark test sentences for all 3 target languages
test_samples = {
    "Marathi (mr)": "नरेंद्र मोदी नवी दिल्लीत राहातात.",
    "Bengali (bn)": "রবীন্দ্রনাথ ঠাকুর কলকাতায় জন্মগ্রহণ করেছিলেন।",
    "Telugu (te)": "హైదరాబాద్ లో అమరరాజా సంస్థ నూతన పరిశోధన కేంద్రాన్ని ఏర్పాటు చేసింది."
}

print("\n🚀 --- LIVE INFERENCE RESULTS --- 🚀\n")

for lang, text in test_samples.items():
    print(f"📌 {lang}")
    print(f"   Text: \"{text}\"")
    results = ner_pipeline(text)

    if not results:
        print("   └─ No entities detected.")
    else:
        for entity in results:
            print(f"   └─ Entity: {entity['word']:<25} | Tag: {entity['entity_group']:<6} | Confidence: {entity['score']:.4f}")
    print("-" * 65)

📥 Loading live production pipeline from Hugging Face Hub: Learner-sai/muril-ner-multilingual...


config.json:   0%|          | 0.00/984 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  948MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/6.41M [00:00<?, ?B/s]


🚀 --- LIVE INFERENCE RESULTS --- 🚀

📌 Marathi (mr)
   Text: "नरेंद्र मोदी नवी दिल्लीत राहातात."
   └─ Entity: नरेंद्र                   | Tag: ORG    | Confidence: 0.9224
   └─ Entity: मोदी नवी                  | Tag: PER    | Confidence: 0.9229
   └─ Entity: राहात                     | Tag: LOC    | Confidence: 0.9937
   └─ Entity: ##ात                      | Tag: LOC    | Confidence: 0.9956
   └─ Entity: .                         | Tag: LOC    | Confidence: 0.9977
-----------------------------------------------------------------
📌 Bengali (bn)
   Text: "রবীন্দ্রনাথ ঠাকুর কলকাতায় জন্মগ্রহণ করেছিলেন।"
   └─ Entity: রবীন্দ্রনাথ               | Tag: ORG    | Confidence: 0.9503
   └─ Entity: ঠাকুর কলকাতায়            | Tag: PER    | Confidence: 0.9048
   └─ Entity: জন্মগ্রহণ                 | Tag: LOC    | Confidence: 0.9992
   └─ Entity: করেছিলেন                  | Tag: LOC    | Confidence: 0.9993
   └─ Entity: ।                         | Tag: LOC    | Confidence: 0.9994
--------------

# Step 8 **bold text**
As model trained previously its misflagged some tags

In [ ]:
import numpy as np
import evaluate
import math
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

# 1. CORRECT NAAMAPADAM LABEL SCHEMA (0 MUST BE 'O')
LABEL_LIST = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}

print(f"✅ Verified Label Schema: {ID2LABEL}")

# 2. Re-initialize Model Head with Correct ID Mapping
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

# 3. Configure Training (Fast 50k subset for ~3 min completion)
SUBSET_SIZE = 50000
train_sub = train_dataset_raw.shuffle(seed=42).select(range(SUBSET_SIZE))

print("⚡ Tokenizing with corrected labels...")
tokenized_train_correct = train_sub.map(tokenize_and_align_labels, batched=True, remove_columns=train_sub.column_names, num_proc=2)
tokenized_val_correct = val_dataset_raw.map(tokenize_and_align_labels, batched=True, remove_columns=val_dataset_raw.column_names, num_proc=2)

BATCH_SIZE = 16
NUM_EPOCHS = 3
TOTAL_STEPS = math.ceil(len(tokenized_train_correct) / BATCH_SIZE) * NUM_EPOCHS
WARMUP_STEPS = int(TOTAL_STEPS * 0.1)

training_args = TrainingArguments(
    output_dir="./muril_ner_corrected",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=32,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    warmup_steps=WARMUP_STEPS,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    hub_token=HF_TOKEN,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_correct,
    eval_dataset=tokenized_val_correct,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("🚀 Launching Retraining with Corrected Label Schema...")
trainer.train()

# Push corrected model weights to HF Hub
trainer.push_to_hub("Corrected label mapping for Multilingual Indic NER")
print("✅ Corrected Model Pushed to Hugging Face Hub!")

✅ Verified Label Schema: {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC'}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ig

⚡ Tokenizing with corrected labels...


Map (num_proc=2):   0%|          | 0/50000 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Corrected Model Pushed to Hugging Face Hub!


In [ ]:
# Test Live Pipeline
ner_pipeline = pipeline(
    task="ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="first"  # Standard subword aggregation for IOB2 tags
)

test_samples = {
    "Marathi (mr)": "नरेंद्र मोदी नवी दिल्लीत राहातात.",
    "Bengali (bn)": "রবীন্দ্রনাথ ঠাকুর কলকাতায় জন্মগ্রহণ করেছিলেন।",
    "Telugu (te)": "హైదరాబాద్ లో అమరరాజా సంస్థ నూతన పరిశోధన కేంద్రాన్ని ఏర్పాటు చేసింది."
}

print("\n✨ --- CORRECTED LIVE INFERENCE RESULTS --- ✨\n")

for lang, text in test_samples.items():
    print(f"📌 {lang}")
    print(f"   Text: \"{text}\"")
    results = ner_pipeline(text)
    for entity in results:
        print(f"   └─ Entity: {entity['word']:<25} | Tag: {entity['entity_group']:<6} | Score: {entity['score']:.4f}")
    print("-" * 65)


✨ --- CORRECTED LIVE INFERENCE RESULTS --- ✨

📌 Marathi (mr)
   Text: "नरेंद्र मोदी नवी दिल्लीत राहातात."
-----------------------------------------------------------------
📌 Bengali (bn)
   Text: "রবীন্দ্রনাথ ঠাকুর কলকাতায় জন্মগ্রহণ করেছিলেন।"
-----------------------------------------------------------------
📌 Telugu (te)
   Text: "హైదరాబాద్ లో అమరరాజా సంస్థ నూతన పరిశోధన కేంద్రాన్ని ఏర్పాటు చేసింది."
-----------------------------------------------------------------


In [ ]:
from transformers import pipeline

# Load your fine-tuned model
HUB_MODEL_ID = f"{HF_USERNAME}/muril-ner-multilingual"

ner_pipeline = pipeline(
    task="ner",
    model=HUB_MODEL_ID,
    tokenizer=HUB_MODEL_ID,
    aggregation_strategy="first"
)

# Expanded Test Suite focusing on Organizations (ORG)
org_test_samples = {
    "Marathi - Corporate & Space Agency":
        "टाटा कन्सल्टन्सी सर्व्हिसेसचे मुख्यालय मुंबईत आहे आणि भारतीय अंतराळ संशोधन संस्था श्रीहरिकोटा येथून काम करते.",

    "Marathi - Banking & Education":
        "स्टेट बँक ऑफ इंडिया आणि सावित्रीबाई फुले पुणे विद्यापीठ यांनी सामंजस्य करार केला.",

    "Bengali - Tech & Space Agency":
        "ভারতীয় মহাকাশ গবেষণা সংস্থা শ্রীহরিকোটা থেকে নতুন রকেট পাঠিয়েছে এবং টাটা স্টিল কলকাতায় বিনিয়োগ করছে।",

    "Bengali - Banking & Government":
        "স্টেট ব্যাংক অব ইন্ডিয়া এবং ভারতীয় রিজার্ভ ব্যাংক ঢাকায় একটি যৌথ সম্মেলন করেছে।",

    "Telugu - Corporate & Space Agency":
        "భారతీయ అంతరిక్ష పరిశోధనా సంస్థ శ్రీహరికోట నుంచి కొత్త రాకెట్‌ను ప్రయోగించింది మరియు టాటా కన్సల్టెన్సీ సర్వీసెస్ హైదరాబాద్‌లో పనిచేస్తోంది.",

    "Telugu - Banking & Education":
        "స్టేట్ బ్యాంక్ ఆఫ్ ఇండియా విజయవాడలో కొత్త శాఖను ప్రారంభించింది."
}

print("🏢 --- TESTING ORGANIZATION (ORG) ENTITY EXTRACTION --- 🏢\n")

for category, text in org_test_samples.items():
    print(f"📌 {category}")
    print(f"   Text: \"{text}\"")
    results = ner_pipeline(text)

    if not results:
        print("   └─ No entities detected.")
    else:
        for entity in results:
            print(f"   └─ Entity: {entity['word']:<35} | Tag: {entity['entity_group']:<6} | Score: {entity['score']:.4f}")
    print("-" * 75)

config.json:   0%|          | 0.00/984 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  948MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🏢 --- TESTING ORGANIZATION (ORG) ENTITY EXTRACTION --- 🏢

📌 Marathi - Corporate & Space Agency
   Text: "टाटा कन्सल्टन्सी सर्व्हिसेसचे मुख्यालय मुंबईत आहे आणि भारतीय अंतराळ संशोधन संस्था श्रीहरिकोटा येथून काम करते."
   └─ Entity: टाटा कन्सल्टन्सी सर्व्हिसेसचे       | Tag: ORG    | Score: 0.8869
   └─ Entity: मुंबईत                              | Tag: LOC    | Score: 0.9120
   └─ Entity: भारतीय अंतराळ संशोधन संस्था         | Tag: ORG    | Score: 0.8793
   └─ Entity: श्रीहरिकोटा                         | Tag: LOC    | Score: 0.8578
---------------------------------------------------------------------------
📌 Marathi - Banking & Education
   Text: "स्टेट बँक ऑफ इंडिया आणि सावित्रीबाई फुले पुणे विद्यापीठ यांनी सामंजस्य करार केला."
   └─ Entity: स्टेट बँक ऑफ इंडिया                 | Tag: ORG    | Score: 0.8791
   └─ Entity: सावित्रीबाई फुले पुणे विद्यापीठ     | Tag: ORG    | Score: 0.8836
---------------------------------------------------------------------------
📌 Bengali - Tech & Space Ag

In [ ]:
# @title 🔍 Indic Multilingual NER Test Form
# @markdown Enter any sentence in **Marathi**, **Bengali**, or **Telugu** below to test entity extraction:

text_input = "सुंदर पिचाई गुगलचा सीईओ आहे" # @param {type:"string"}

from transformers import pipeline

HUB_MODEL_ID = f"{HF_USERNAME}/muril-ner-multilingual"

# 1. Load pipeline if not already loaded
if 'ner_pipeline' not in globals():
    print(f"📥 Loading model: {HUB_MODEL_ID}...")
    ner_pipeline = pipeline(
        task="ner",
        model=HUB_MODEL_ID,
        tokenizer=HUB_MODEL_ID,
        aggregation_strategy="first"
    )

# 2. Run prediction
results = ner_pipeline(text_input)

# 3. Display results formatted nicely
print("\n" + "="*50)
print(f"📝 Input Text: \"{text_input}\"")
print("="*50)

if not results:
    print("ℹ️ No entities detected (All words categorized as Outside 'O').")
else:
    print(f"{'Extracted Entity':<30} | {'Tag':<8} | {'Confidence Accuracy':<15}")
    print("-" * 60)
    for entity in results:
        word = entity['word']
        tag = entity['entity_group']
        score = entity['score'] * 100
        print(f"{word:<30} | {tag:<8} | {score:.2f}%")
print("="*50)


📝 Input Text: "सुंदर पिचाई गुगलचा सीईओ आहे"
Extracted Entity               | Tag      | Confidence Accuracy
------------------------------------------------------------
सुंदर पिचाई                    | PER      | 92.42%
गुगलचा                         | ORG      | 89.51%


In [ ]:
# Save current working pipeline weights locally to Google Drive / Colab memory
from transformers import AutoModelForTokenClassification, AutoTokenizer

model_id = "Learner-sai/muril-ner-multilingual"

# Download current best model
backup_model = AutoModelForTokenClassification.from_pretrained(model_id)
backup_tokenizer = AutoTokenizer.from_pretrained(model_id)

# Save to a dedicated backup folder
backup_model.save_pretrained("./backup_best_model")
backup_tokenizer.save_pretrained("./backup_best_model")

print("🔒 Backup locked in local folder './backup_best_model'!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🔒 Backup locked in local folder './backup_best_model'!


Improving model

In [ ]:
import warnings
import torch
import numpy as np
from datasets import load_dataset, concatenate_datasets
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

# Suppress pipeline warning messages
warnings.filterwarnings("ignore", category=UserWarning, module="transformers.pipelines.base")

# ==========================================
# STEP 1: DEFINE MODEL VERSION & LABELS
# ==========================================
# ⚠️ Using a NEW repository name to protect your v1 model!
NEW_HUB_MODEL_ID = "Learner-sai/muril-ner-multilingual-v2"
BASE_MODEL_NAME = "google/muril-base-cased"
LOCAL_OUTPUT_DIR = "./muril_ner_v2_results"

LABEL_LIST = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
label2id = {label: i for i, label in enumerate(LABEL_LIST)}
id2label = {i: label for i, label in enumerate(LABEL_LIST)}

# ==========================================
# STEP 2: LOAD TOKENIZER & DATASETS
# ==========================================
print("📥 Loading Tokenizer and Datasets...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

# Load Indic languages from Naamapadam (bn, mr, te)
ds_bn = load_dataset("ai4bharat/naamapadam", "bn")
ds_mr = load_dataset("ai4bharat/naamapadam", "mr")
ds_te = load_dataset("ai4bharat/naamapadam", "te")

# Combine datasets
raw_train = concatenate_datasets([ds_bn["train"], ds_mr["train"], ds_te["train"]])
raw_eval = concatenate_datasets([ds_bn["validation"], ds_mr["validation"], ds_te["validation"]])

# Select dataset split size (e.g., 100,000 train / 10,000 eval)
train_dataset = raw_train.shuffle(seed=42).select(range(100000))
eval_dataset = raw_eval.shuffle(seed=42).select(range(len(raw_eval)))

# ==========================================
# STEP 3: SUBWORD LABEL ALIGNMENT FUNCTION
# ==========================================
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

print("⚡ Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True, remove_columns=raw_train.column_names)
tokenized_eval = eval_dataset.map(tokenize_and_align_labels, batched=True, remove_columns=raw_eval.column_names)

# ==========================================
# STEP 4: SETUP METRICS & INITIALIZE MODEL
# ==========================================
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [LABEL_LIST[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [LABEL_LIST[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

print("🏗️ Initializing Model...")
model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id
)

# ==========================================
# STEP 5: TRAINER CONFIGURATION (DIRECT HUB PUSH)
# ==========================================
training_args = TrainingArguments(
    output_dir=LOCAL_OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),        # Mixed precision for fast GPU execution
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=True,                      # Direct Hugging Face integration
    hub_model_id=NEW_HUB_MODEL_ID,        # Pushes to v2 repo
    logging_steps=500
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

# ==========================================
# STEP 6: TRAIN & PUSH DIRECTLY TO HUB
# ==========================================
print("🚀 Launching Retraining...")
trainer.train()

print("💾 Pushing best checkpoint directly to Hugging Face Hub...")
trainer.push_to_hub(commit_message="Pushed v2 multilingual NER model checkpoint")
print(f"✅ Success! Your model is live on Hugging Face at https://huggingface.co/{NEW_HUB_MODEL_ID}")

📥 Loading Tokenizer and Datasets...
⚡ Tokenizing datasets...
🏗️ Initializing Model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ig

🚀 Launching Retraining...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.212797,0.220784,0.748410,0.770699,0.759391,0.936100
2,0.183963,0.217274,0.737491,0.795044,0.765187,0.936586


...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.212797,0.220784,0.748410,0.770699,0.759391,0.936100
2,0.183963,0.217274,0.737491,0.795044,0.765187,0.936586
3,0.143468,0.221317,0.750074,0.787197,0.768187,0.938344


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Pushing best checkpoint directly to Hugging Face Hub...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Success! Your model is live on Hugging Face at https://huggingface.co/Learner-sai/muril-ner-multilingual-v2


In [ ]:
import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    DataCollatorForTokenClassification
)

# 1. Load YOUR FINE-TUNED model from Hugging Face Hub (NOT the base model!)
MODEL_ID = "Learner-sai/muril-ner-multilingual"

print(f"📥 Loading Fine-Tuned Model weights from: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForTokenClassification.from_pretrained(MODEL_ID)

LABEL_LIST = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
seqeval = evaluate.load("seqeval")

# Subword alignment function
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [LABEL_LIST[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [LABEL_LIST[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Re-instantiate Trainer with the loaded fine-tuned model
trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

# 2. Evaluate each language individually
languages = ["bn", "mr", "te"]
per_lang_results = []

print("\n🔍 Evaluating Fine-Tuned Model Per Language...\n")

for lang in languages:
    print(f"⌛ Evaluating {lang.upper()} dataset...")

    # Load dataset split
    raw_lang_eval = load_dataset("ai4bharat/naamapadam", lang, split="validation")

    # Tokenize
    tokenized_lang_eval = raw_lang_eval.map(
        tokenize_and_align_labels,
        batched=True,
        remove_columns=raw_lang_eval.column_names
    )

    # Evaluate
    eval_metrics = trainer.evaluate(eval_dataset=tokenized_lang_eval)

    per_lang_results.append({
        "Language": f"{lang.upper()} ({'Bengali' if lang=='bn' else 'Marathi' if lang=='mr' else 'Telugu'})",
        "Precision": round(eval_metrics.get("eval_precision", 0), 4),
        "Recall": round(eval_metrics.get("eval_recall", 0), 4),
        "F1 Score": round(eval_metrics.get("eval_f1", 0), 4),
        "Accuracy": round(eval_metrics.get("eval_accuracy", 0), 4),
        "Eval Loss": round(eval_metrics.get("eval_loss", 0), 4)
    })

# 3. Print clean table
df_results = pd.DataFrame(per_lang_results)
print("\n" + "="*60)
print("📊 PER-LANGUAGE EVALUATION RESULTS (FINE-TUNED MODEL)")
print("="*60)
print(df_results.to_string(index=False))

📥 Loading Fine-Tuned Model weights from: Learner-sai/muril-ner-multilingual...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


🔍 Evaluating Fine-Tuned Model Per Language...

⌛ Evaluating BN dataset...


Map:   0%|          | 0/4859 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
No log,0.208219,0,0.740788,0.774721,0.757375,0.942312


...

Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
No log,0.228990,0,0.747977,0.792468,0.769580,0.932108


⌛ Evaluating TE dataset...


Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
No log,0.266839,0,0.724162,0.767271,0.745093,0.918768



📊 PER-LANGUAGE EVALUATION RESULTS (FINE-TUNED MODEL)
    Language  Precision  Recall  F1 Score  Accuracy  Eval Loss
BN (Bengali)     0.7408  0.7747    0.7574    0.9423     0.2082
MR (Marathi)     0.7480  0.7925    0.7696    0.9321     0.2290
 TE (Telugu)     0.7242  0.7673    0.7451    0.9188     0.2668


In [ ]:
import random
import pandas as pd
from seqeval.metrics import classification_report, accuracy_score, f1_score

# List of NER tags matching your schema
LABELS = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
LANGUAGES = ["Bengali", "Marathi", "Telugu"]

def generate_mock_ner_data(num_sentences=1000, sentence_length=15):
    """Generates synthetic y_true and y_pred token label lists."""
    y_true_all, y_pred_all = [], []

    for _ in range(num_sentences):
        sentence_true, sentence_pred = [], []
        for _ in range(sentence_length):
            # 80% chance of 'O', 20% chance of an entity tag
            true_label = "O" if random.random() < 0.8 else random.choice(LABELS[1:])

            # Simulate a model with ~85% accuracy on entities
            if random.random() < 0.85:
                pred_label = true_label
            else:
                pred_label = random.choice(LABELS)

            sentence_true.append(true_label)
            sentence_pred.append(pred_label)

        y_true_all.append(sentence_true)
        y_pred_all.append(sentence_pred)

    return y_true_all, y_pred_all

# Run mock evaluation across target languages
mock_summary = []

for lang in LANGUAGES:
    y_true, y_pred = generate_mock_ner_data()

    # Calculate exact span-level metrics
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)

    mock_summary.append({
        "Language": lang,
        "Mock F1 Score": round(f1, 4),
        "Mock Accuracy": round(acc, 4)
    })

df = pd.DataFrame(mock_summary)
print("📊 MOCK PIPELINE EVALUATION OUTPUT:")
print(df.to_string(index=False))

📊 MOCK PIPELINE EVALUATION OUTPUT:
Language  Mock F1 Score  Mock Accuracy
 Bengali         0.6885         0.8735
 Marathi         0.6898         0.8695
  Telugu         0.6930         0.8719


In [ ]:
import torch
import evaluate
import numpy as np
import pandas as pd
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification
)

# ==========================================
# 1. MODEL & TOKENIZER CONFIGURATION
# ==========================================
# Explicitly pointing to the v2 model repository
MODEL_ID = "Learner-sai/muril-ner-multilingual-v2"
LANGUAGES = ["bn", "mr", "te"]  # Bengali, Marathi, Telugu

LABEL_LIST = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device set to: {device}")

print(f"📥 Loading model weights & tokenizer from: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(device)

# Ensure evaluation mode (disables dropout layers)
model.eval()

# Load seqeval metric for span-level entity evaluation
seqeval = evaluate.load("seqeval")

# ==========================================
# 2. SUBWORD ALIGNMENT FUNCTION
# ==========================================
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=256  # Increased length to ensure no entities are truncated
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# ==========================================
# 3. EVALUATION FUNCTION FOR SINGLE LANGUAGE
# ==========================================
def evaluate_language(lang_code, batch_size=32):
    raw_eval = load_dataset("ai4bharat/naamapadam", lang_code, split="validation")

    tokenized_eval = raw_eval.map(
        tokenize_and_align_labels,
        batched=True,
        remove_columns=raw_eval.column_names,
        desc=f"Tokenizing {lang_code.upper()}"
    )

    collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
    eval_dataloader = DataLoader(tokenized_eval, batch_size=batch_size, collate_fn=collator)

    all_preds, all_labels = [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in eval_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()

            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1).cpu().numpy()
            labels_cpu = labels.cpu().numpy()

            for pred, label in zip(predictions, labels_cpu):
                true_pred = [LABEL_LIST[p] for (p, l) in zip(pred, label) if l != -100]
                true_label = [LABEL_LIST[l] for (p, l) in zip(pred, label) if l != -100]
                all_preds.append(true_pred)
                all_labels.append(true_label)

    results = seqeval.compute(predictions=all_preds, references=all_labels)
    avg_loss = total_loss / len(eval_dataloader)

    lang_name = "Bengali" if lang_code == "bn" else "Marathi" if lang_code == "mr" else "Telugu"

    return {
        "Language": f"{lang_name} ({lang_code.upper()})",
        "Strict F1": round(results["overall_f1"], 4),
        "Precision": round(results["overall_precision"], 4),
        "Recall": round(results["overall_recall"], 4),
        "Accuracy": round(results["overall_accuracy"], 4),
        "Eval Loss": round(avg_loss, 4)
    }

# ==========================================
# 4. EXECUTE & DISPLAY SUMMARY
# ==========================================
print("\n🔍 Evaluating Learner-sai/muril-ner-multilingual-v2...\n")
results_list = []

for lang in LANGUAGES:
    metrics = evaluate_language(lang)
    results_list.append(metrics)

df_summary = pd.DataFrame(results_list)

print("\n" + "="*65)
print("📊 MODEL V2 PER-LANGUAGE BENCHMARK RESULTS")
print("="*65)
print(df_summary.to_string(index=False))

⚡ Device set to: cuda
📥 Loading model weights & tokenizer from: Learner-sai/muril-ner-multilingual-v2...


config.json:   0%|          | 0.00/984 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/6.41M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  948MB            

...


🔍 Evaluating Learner-sai/muril-ner-multilingual-v2...



Tokenizing BN:   0%|          | 0/4859 [00:00<?, ? examples/s]

Tokenizing MR:   0%|          | 0/2300 [00:00<?, ? examples/s]

Tokenizing TE:   0%|          | 0/2700 [00:00<?, ? examples/s]


📊 MODEL V2 PER-LANGUAGE BENCHMARK RESULTS
    Language  Strict F1  Precision  Recall  Accuracy  Eval Loss
Bengali (BN)     0.7708     0.7534  0.7890    0.9448     0.2053
Marathi (MR)     0.7751     0.7554  0.7958    0.9356     0.2231
 Telugu (TE)     0.7565     0.7382  0.7758    0.9232     0.2627


In [ ]:
import warnings
import torch
import numpy as np
from datasets import load_dataset, concatenate_datasets
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

warnings.filterwarnings("ignore", category=UserWarning, module="transformers.pipelines.base")

# ==========================================
# STEP 1: DEFINE MODEL VERSION & LABELS
# ==========================================
NEW_HUB_MODEL_ID = "Learner-sai/muril-ner-multilingual-v3"
BASE_MODEL_NAME = "google/muril-base-cased"
LOCAL_OUTPUT_DIR = "./muril_ner_v3_results"

LABEL_LIST = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
label2id = {label: i for i, label in enumerate(LABEL_LIST)}
id2label = {i: label for i, label in enumerate(LABEL_LIST)}

# ==========================================
# STEP 2: LOAD TOKENIZER & FULL DATASETS
# ==========================================
print("📥 Loading Tokenizer and Datasets...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

ds_bn = load_dataset("ai4bharat/naamapadam", "bn")
ds_mr = load_dataset("ai4bharat/naamapadam", "mr")
ds_te = load_dataset("ai4bharat/naamapadam", "te")

# Combine full datasets without .select(range(100000))!
raw_train = concatenate_datasets([ds_bn["train"], ds_mr["train"], ds_te["train"]])
raw_eval = concatenate_datasets([ds_bn["validation"], ds_mr["validation"], ds_te["validation"]])

train_dataset = raw_train.shuffle(seed=42)
eval_dataset = raw_eval.shuffle(seed=42)

# ==========================================
# STEP 3: SUBWORD LABEL ALIGNMENT (256 MAX LENGTH)
# ==========================================
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=256  # 👈 Increased from 128 to capture full sentences!
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

print("⚡ Tokenizing datasets...")
tokenized_train = train_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_train.column_names,
    num_proc=4  # Fast parallel tokenization
)
tokenized_eval = eval_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_eval.column_names,
    num_proc=4
)

# ==========================================
# STEP 4: METRICS & MODEL SETUP
# ==========================================
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [LABEL_LIST[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [LABEL_LIST[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

print("🏗️ Initializing Model...")
model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id
)

# ==========================================
# STEP 5: OPTIMIZED TRAINER CONFIGURATION
# ==========================================
training_args = TrainingArguments(
    output_dir=LOCAL_OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,                  # 👈 Optimal learning rate for full dataset
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,       # 👈 Effective batch size = 32
    warmup_ratio=0.1,                    # 👈 10% learning rate warmup prevents early overfitting
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=True,
    hub_model_id=NEW_HUB_MODEL_ID,
    logging_steps=1000
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

# ==========================================
# STEP 6: TRAIN & PUSH v3 TO HUB
# ==========================================
print("🚀 Launching v3 Retraining...")
trainer.train()

print("💾 Pushing best v3 checkpoint directly to Hugging Face Hub...")
trainer.push_to_hub(commit_message="Pushed v3 full-dataset multilingual NER model checkpoint")
print(f"✅ Success! Your v3 model is live at https://huggingface.co/{NEW_HUB_MODEL_ID}")

📥 Loading Tokenizer and Datasets...
⚡ Tokenizing datasets...


Map (num_proc=4):   0%|          | 0/1924668 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/9859 [00:00<?, ? examples/s]

🏗️ Initializing Model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ig

🚀 Launching v3 Retraining...


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
